# 03 — Ingeniería de Características

Construimos la **matriz de features final** lista para los modelos de ML.

Pipeline:
1. Calcular features endógenas de precio (retornos, medias móviles, volatilidad).
2. Construir la variable objetivo `label` (1=precio sube mañana, 0=baja).
3. Limpiar filas con NaN (warmup de ventanas deslizantes).
4. Guardar la matriz de features en `data/processed/`.

Salidas:
- `data/processed/features_all.csv` → features numéricas + label (todos los mercados)

In [1]:
import sys
from pathlib import Path

ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(exist_ok=True)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import build_market_features, build_labels, temporal_split

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

## 1. Cargar datos crudos

In [3]:
price_files = sorted(RAW_DIR.glob("prices_*.csv"))
print(f"Archivos de precios: {len(price_files)}")

price_dfs = {}
for f in price_files:
    slug = f.stem.replace("prices_", "")
    df = pd.read_csv(f, parse_dates=["date"])
    price_dfs[slug] = df

print("Mercados cargados:", list(price_dfs.keys()))

Archivos de precios : 10
Archivos de noticias: 10


## 2. Features endógenas de precio + labels

In [4]:
MARKET_FEATURE_COLS = ["ret_1d", "ret_3d", "ret_7d", "ma7", "ma14", "ma_ratio", "vol7", "price"]

feat_frames = []

for slug, df in price_dfs.items():
    # 1. Market features
    df_feat = build_market_features(df)

    # 2. Labels
    df_feat = build_labels(df_feat)

    # 3. Attach slug
    df_feat["slug"] = slug

    feat_frames.append(df_feat)

prices_all = pd.concat(feat_frames, ignore_index=True)
print(f"Total filas (todos los mercados, con NaN): {len(prices_all)}")
prices_all[MARKET_FEATURE_COLS + ["label", "slug"]].describe()

Total filas (todos los mercados, con NaN): 1647


,ret_1d,ret_3d,ret_7d,ma7,ma14,ma_ratio,vol7,price,label
count,1637.000000,1617.000000,1577.000000,1627.000000,1607.000000,1607.000000,1617.000000,1647.000000,1647.0
mean,-0.017138,-0.034392,-0.064253,0.120038,0.120119,0.973454,0.131907,0.119837,0.264724
std,0.211181,0.257811,0.330336,0.285104,0.284266,0.108432,0.136209,0.285660,0.44132
min,-2.197207,-1.945893,-2.397877,0.000929,0.001071,0.444442,0.000000,0.000500,0.0
25%,-0.027399,-0.095310,-0.200670,0.003786,0.003929,0.917615,0.043280,0.003500,0.0
50%,0.000000,0.000000,0.000000,0.007500,0.007714,0.994708,0.114599,0.007500,0.0
75%,0.009390,0.030153,0.066691,0.024393,0.024740,1.026217,0.173423,0.024000,1.0
max,1.312184,1.312184,1.178652,0.992643,0.989536,1.422676,1.106606,0.993500,1.0


## 3. Limpieza: eliminar filas con NaN en features o label

In [5]:
FEATURE_COLS = MARKET_FEATURE_COLS

before = len(prices_all)
data_clean = prices_all.dropna(subset=FEATURE_COLS + ["label"]).reset_index(drop=True)
data_clean["label"] = data_clean["label"].astype(int)

print(f"Filas antes de limpiar  : {before}")
print(f"Filas después de limpiar: {len(data_clean)}")
print(f"Balance de clases:\n{data_clean['label'].value_counts()}")

Mercados con datos de sentimiento: ['will-the-patriots-win-super-bowl-2025']


In [6]:
SAVE_COLS = ["date", "slug"] + FEATURE_COLS + ["label"]
data_clean[SAVE_COLS].to_csv(PROC_DIR / "features_all.csv", index=False)
print(f"Dataset guardado en data/processed/features_all.csv")
print(f"Shape: {data_clean[SAVE_COLS].shape}")
data_clean[SAVE_COLS].head()

Total filas después del merge: 1647
Columnas: ['date', 'price', 'slug', 'question', 'ret_1d', 'ret_3d', 'ret_7d', 'ma7', 'ma14', 'ma_ratio', 'vol7', 'label', 'gdelt_tone', 'article_count']


## 4. Correlación entre features

In [7]:
corr = data_clean[FEATURE_COLS + ["label"]].corr()

plt.figure(figsize=(10, 8))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    square=True,
)
plt.title("Matriz de correlación — features vs label")
plt.tight_layout()
plt.savefig(PROC_DIR / "correlation_matrix.png", bbox_inches="tight")
plt.show()

Filas antes de limpiar  : 1647
Filas después de limpiar: 1577
Balance de clases:
label
0    1159
1     418
Name: count, dtype: int64


## 5. Verificación del split temporal

In [8]:
train, test = temporal_split(data_clean, train_ratio=0.70)

print(f"Train: {len(train)} filas  | {train['date'].min()} → {train['date'].max()}")
print(f"Test : {len(test)}  filas  | {test['date'].min()} → {test['date'].max()}")
print(f"\nBalance de clases en train:\n{train['label'].value_counts()}")
print(f"\nBalance de clases en test:\n{test['label'].value_counts()}")

Corpus de texto: 18 filas (mercado × día)


TypeError: Object of type int64 is not JSON serializable